over fitting chek through  train and testing deffernt  models

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score



In [2]:
df=pd.read_csv("50_Startups.csv")
print(df.shape)
print(df.head())
print(df.info())

(50, 5)
   R&D Spend  Administration  Marketing Spend       State     Profit
0  165349.20       136897.80        471784.10    New York  192261.83
1  162597.70       151377.59        443898.53  California  191792.06
2  153441.51       101145.55        407934.54     Florida  191050.39
3  144372.41       118671.85        383199.62    New York  182901.99
4  142107.34        91391.77        366168.42     Florida  166187.94
<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   R&D Spend        50 non-null     float64
 1   Administration   50 non-null     float64
 2   Marketing Spend  50 non-null     float64
 3   State            50 non-null     str    
 4   Profit           50 non-null     float64
dtypes: float64(4), str(1)
memory usage: 2.5 KB
None


In [3]:
x=df.drop("Profit",axis=1)
y=df["Profit"]
print(x.shape,y.shape)

(50, 4) (50,)


In [4]:
x= pd.get_dummies(x, columns=["State"], drop_first=True, dtype=int)
print(x.head())

   R&D Spend  Administration  Marketing Spend  State_Florida  State_New York
0  165349.20       136897.80        471784.10              0               1
1  162597.70       151377.59        443898.53              0               0
2  153441.51       101145.55        407934.54              1               0
3  144372.41       118671.85        383199.62              0               1
4  142107.34        91391.77        366168.42              1               0


In [5]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)
print(x_train.shape, x_test.shape)

(40, 5) (10, 5)


In [6]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(x_train, y_train)

print("Linear Train R2:", r2_score(y_train, lr.predict(x_train)))
print("Linear Test R2:", r2_score(y_test, lr.predict(x_test)))

Linear Train R2: 0.9537019995248526
Linear Test R2: 0.8987266414328636


In [8]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(random_state=42)
dt.fit(x_train, y_train)

print("Tree Train R2:", r2_score(y_train, dt.predict(x_train)))
print("Tree Test R2:", r2_score(y_test, dt.predict(x_test)))

Tree Train R2: 1.0
Tree Test R2: 0.835899909497648


In [9]:
for d in [2, 3, 4, 5, None]:
    dt = DecisionTreeRegressor(max_depth=d, random_state=42)
    dt.fit(x_train, y_train)
    print(d, "Train:", r2_score(y_train, dt.predict(x_train)),
             "Test:", r2_score(y_test, dt.predict(x_test)))

2 Train: 0.8933481909411258 Test: 0.7226360781651671
3 Train: 0.9686293734459094 Test: 0.8001150060564005
4 Train: 0.9876876614905055 Test: 0.7153378786023552
5 Train: 0.9992097709051426 Test: 0.8014814646479143
None Train: 1.0 Test: 0.835899909497648


In [10]:
from sklearn.model_selection import cross_val_score

models = {
    "Linear": lr,
    "Tree depth=3": DecisionTreeRegressor(max_depth=3, random_state=42),
    "Tree None": DecisionTreeRegressor(random_state=42),
}

for name, m in models.items():
    scores = cross_val_score(m, x, y, cv=5, scoring="r2")
    print(name, "Mean R2:", scores.mean(), "Std:", scores.std())

Linear Mean R2: -0.25513996551976664 Std: 0.7504085317778573
Tree depth=3 Mean R2: -6.678015749419285 Std: 4.84393447961696
Tree None Mean R2: -5.741402388505805 Std: 4.229852773848366


In [11]:
from sklearn.model_selection import cross_val_score, KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Linear": lr,
    "Tree depth=3": DecisionTreeRegressor(max_depth=3, random_state=42),
    "Tree None": DecisionTreeRegressor(random_state=42),
}

for name, m in models.items():
    scores = cross_val_score(m, x, y, cv=kf, scoring="r2")
    print(name, "Mean R2:", scores.mean(), "Std:", scores.std())

Linear Mean R2: 0.9278733909482193 Std: 0.04375818245689391
Tree depth=3 Mean R2: 0.850881564761664 Std: 0.06173138033982451
Tree None Mean R2: 0.8719067313002397 Std: 0.07174732081455425


In [12]:
print(y.head(10))


0    192261.83
1    191792.06
2    191050.39
3    182901.99
4    166187.94
5    156991.12
6    156122.51
7    155752.60
8    152211.77
9    149759.96
Name: Profit, dtype: float64


In [13]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42)
scores = cross_val_score(rf, x, y, cv=kf, scoring="r2")
print("Random Forest Mean R2:", scores.mean(), "Std:", scores.std())

Random Forest Mean R2: 0.9276542334648774 Std: 0.04190465701522188
